# 🎬 Netflix AI Design Studio
### Course-End Project — Creating Designs by Leveraging OpenAI & Gradio UI

---

## 📋 Project Overview

This notebook demonstrates a **production-ready AI image generation platform** for Netflix creative campaigns.
Using **OpenAI DALL-E** and **Gradio UI**, the platform:

- 🖼️ Generates bespoke banner & poster images from text prompts
- 🎨 Provides Netflix-themed style presets for consistent branding
- 📐 Supports multiple output sizes (Banner, Poster, Square)
- 💾 Enables one-click image download
- 🤖 Auto-detects best available model (DALL-E 3 → DALL-E 2 fallback)
- 🔒 Handles API keys securely via Colab Secrets

---

## 🏗️ Architecture

```
User Prompt → Prompt Enhancer → OpenAI DALL-E API
                                        ↓
                             Generated Image URL
                                        ↓
                          PIL Image Processing & Display
                                        ↓
                               Gradio Output Panel
```

In [ ]:
# STEP 1 — Install all required libraries
!pip install -q openai gradio pillow requests
print('✅ All dependencies installed successfully!')

---
## 🔑 Step 2 — Configure API Key (Securely)

Your API key is **never hardcoded**. It is read from:
1. **Google Colab Secrets** *(recommended)* — click the 🔑 key icon on the left sidebar, add `OPENAI_API_KEY`
2. **Environment variable** — `export OPENAI_API_KEY=sk-...`
3. **Manual input** — masked prompt, not stored anywhere

In [ ]:
# STEP 2 — Secure API Key Configuration
import os
import getpass

def load_api_key():
    # Priority 1: Google Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get('OPENAI_API_KEY')
        if key:
            print('🔐 API key loaded from Google Colab Secrets.')
            return key
    except Exception:
        pass
    # Priority 2: Environment variable
    key = os.environ.get('OPENAI_API_KEY', '')
    if key:
        print('🔐 API key loaded from environment variable.')
        return key
    # Priority 3: Masked manual entry
    print('⚠️  No API key found automatically.')
    print('👉  TIP: Add it to Colab Secrets as OPENAI_API_KEY for future runs.')
    key = getpass.getpass('🔑 Enter your OpenAI API key (input is hidden): ')
    return key

OPENAI_API_KEY = load_api_key()

if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith('sk-'):
    raise ValueError('❌ Invalid or missing OpenAI API key. Please check and try again.')

print(f'✅ API key configured. (Starts with: {OPENAI_API_KEY[:8]}...)')

---
## 📦 Step 3 — Import Libraries

In [ ]:
# STEP 3 — Import all required libraries
import gradio as gr
import requests
from PIL import Image
from io import BytesIO
from openai import OpenAI
import time
import os

client = OpenAI(api_key=OPENAI_API_KEY)

print('✅ All libraries imported and OpenAI client initialised!')
print(f'   Gradio version : {gr.__version__}')
print(f'   OpenAI version : {__import__("openai").__version__}')

---
## 🤖 Step 4 — Auto-Detect Available Model

This cell checks whether your account has **DALL-E 3** access.
If not, it automatically falls back to **DALL-E 2** — both work with this notebook.

| Model | Quality | Max Size | Cost/image |
|-------|---------|----------|------------|
| DALL-E 3 | Much better | 1792×1024 | $0.04–$0.12 |
| DALL-E 2 | Good | 1024×1024 | $0.016–$0.020 |

In [ ]:
# STEP 4 — Auto-detect best available DALL-E model

def detect_best_model():
    """Try DALL-E 3 first; fall back to DALL-E 2 if not available."""
    # Try a minimal DALL-E 3 call (1x1 placeholder — cheapest possible)
    try:
        test = client.images.generate(
            model='gpt-image-1',
            prompt='a red circle',
            size='1024x1024',
            quality='standard',
            n=1,
        )
        print('✅ DALL-E 3 is available on your account!')
        return 'dall-e-3'
    except Exception as e:
        err = str(e).lower()
        if 'does not exist' in err or 'invalid_value' in err or 'model' in err:
            print('⚠️  DALL-E 3 is NOT available on your account (requires paid tier).')
            print('✅ Falling back to gpt-image-1fully supported on free/trial accounts.')
            return 'gpt-image-1'
        else:
            # Some other error (quota, key issue etc) — default to dall-e-2 safely
            print(f'⚠️  Could not verify DALL-E 3 ({e}). Defaulting to gpt-image-1.')
            return 'gpt-image-1'

ACTIVE_MODEL = detect_best_model()
print(f'\n🎯 Active model: {ACTIVE_MODEL}')

# DALL-E 2 only supports square sizes; DALL-E 3 supports wide/tall
if ACTIVE_MODEL == 'dall-e-2':
    print('ℹ️  Note: DALL-E 2 only supports square output (1024x1024).')
    print('   All size selections will be set to 1024x1024 automatically.')

---
## 🧠 Step 5 — Core Logic: `generate_image` Function

In [ ]:
# STEP 5 — Core image generation logic

STYLE_PRESETS = {
    '🎬 Netflix Original (Cinematic)': (
        'cinematic Netflix poster style, dramatic lighting, high contrast, '
        'deep shadows, bold typography space, professional film photography, '
        '4K ultra-realistic, dark atmospheric background'
    ),
    '🌆 Neo-Noir Thriller': (
        'neo-noir style, moody blue-purple tones, rain-slicked streets, '
        'neon reflections, high contrast shadows, detective drama aesthetic'
    ),
    '🚀 Sci-Fi Epic': (
        'sci-fi epic style, futuristic space environment, glowing neon elements, '
        'alien landscapes, cosmic background, ultra-detailed CGI render'
    ),
    '👑 Fantasy & Adventure': (
        'epic fantasy style, magical golden lighting, mythical creatures, '
        'lush ancient worlds, painterly illustration, sweeping vistas'
    ),
    '💘 Romantic Drama': (
        'romantic drama style, warm golden-hour lighting, soft bokeh background, '
        'emotional depth, pastel tones, premium lifestyle photography'
    ),
    '😂 Comedy & Fun': (
        'vibrant comedy poster style, bright saturated colors, playful typography '
        'space, fun energetic composition, bold graphic design'
    ),
    '🎨 Minimalist Modern': (
        'minimalist modern design, clean negative space, geometric shapes, '
        'flat design aesthetic, muted palette with one bold accent color'
    ),
}

# DALL-E 3 supports rectangular sizes; DALL-E 2 is square-only
SIZE_MAP_DALLE3 = {
    '🖼️  Poster (1024x1792 - Vertical)' : '1024x1792',
    '🎞️  Banner (1792x1024 - Horizontal)': '1792x1024',
    '⬛  Square  (1024x1024)'             : '1024x1024',
}
SIZE_MAP_DALLE2 = {
    '🖼️  Poster (1024x1792 - Vertical)' : '1024x1024',
    '🎞️  Banner (1792x1024 - Horizontal)': '1024x1024',
    '⬛  Square  (1024x1024)'             : '1024x1024',
}

QUALITY_MAP = {
    '⚡ Standard (Faster)' : 'medium',
    '🏆 HD (Best Quality)' : 'high',
}


def build_enhanced_prompt(user_prompt, style_key):
    style_suffix = STYLE_PRESETS.get(style_key, '')
    return (
        f'{user_prompt.strip()}. '
        f'Visual style: {style_suffix}. '
        'No watermarks, no text overlays. Professional commercial quality.'
    )


def generate_image(prompt, style_key, size_key, quality_key):
    if not prompt or not prompt.strip():
        return None, '❌ Please enter a prompt before generating.'
    if len(prompt.strip()) < 5:
        return None, '❌ Prompt is too short. Please describe the image in more detail.'

    enhanced_prompt = build_enhanced_prompt(prompt, style_key)

    # Pick correct size map based on active model
    size_map   = SIZE_MAP_DALLE3 if ACTIVE_MODEL == 'dall-e-3' else SIZE_MAP_DALLE2
    size_value = size_map.get(size_key, '1024x1024')

    # DALL-E 2 does not support quality param
    quality_value = QUALITY_MAP.get(quality_key, 'standard')

    status_lines = [
        f'🤖 Model    : {ACTIVE_MODEL}',
        f'🎨 Style    : {style_key}',
        f'📐 Size     : {size_value}',
        f'⚙️  Quality  : {quality_value if ACTIVE_MODEL == "gpt-image-1" else "N/A (DALL-E 2)"}',
        f'✍️  Prompt   : {prompt[:120]}{"..." if len(prompt) > 120 else ""}',
        '',
        '⏳ Calling API...',
    ]

    try:
        if ACTIVE_MODEL == 'gpt-image-1':
            response = client.images.generate(
                model=ACTIVE_MODEL,
                prompt=enhanced_prompt,
                size=size_value,
                quality=quality_value,
                n=1,
            )
        else:

            response = client.images.generate(
            model=ACTIVE_MODEL,
            prompt=enhanced_prompt,
            size=size_value,
            quality=quality_value,
            n=1,
            response_format='b64_json'
            )
    except Exception as e:
        error_msg = str(e)
        if 'content_policy' in error_msg.lower():
            return None, '⚠️ Prompt rejected by content policy. Please revise your prompt.'
        if 'insufficient_quota' in error_msg.lower():
            return None, '⚠️ OpenAI quota exceeded. Check billing at platform.openai.com.'
        if 'invalid_api_key' in error_msg.lower():
            return None, '❌ Invalid API key. Please re-run Step 2 with a valid key.'
        return None, f'❌ API Error: {error_msg}'

    import base64
    revised_prompt = getattr(response.data[0], 'revised_prompt', None) or 'N/A'

    try:
        b64 = response.data[0].b64_json
        if b64:
            image = Image.open(BytesIO(base64.b64decode(b64))).convert('RGB')
        else:
            image_url = response.data[0].url
            img_resp = requests.get(image_url, timeout=30)
            img_resp.raise_for_status()
            image = Image.open(BytesIO(img_resp.content)).convert('RGB')
    except Exception as e:
        return None, f'❌ Failed to decode image: {e}'
    status_lines += [
        '✅ Image generated successfully!',
        f'📏 Dimensions : {image.width} x {image.height} px',
        f'🤖 Note: {revised_prompt[:200]}{"..." if len(revised_prompt) > 200 else ""}',
    ]

    return image, '\n'.join(status_lines)


print('✅ Core functions defined successfully!')
print(f'   Available styles : {len(STYLE_PRESETS)}')

---
## 🖥️ Step 6 — Build & Launch the Gradio Interface

> ℹ️ After running this cell, a **public `gradio.live` link** appears below — active for 72 hours.

In [ ]:
# STEP 6 — Build the Gradio Interface

CUSTOM_CSS = """
body, .gradio-container {
    font-family: 'Segoe UI', Arial, sans-serif !important;
    background: #141414 !important;
    color: #FFFFFF !important;
}
.app-header {
    text-align: center;
    padding: 32px 16px 16px;
    background: linear-gradient(135deg, #1a0000 0%, #141414 60%);
    border-bottom: 2px solid #E50914;
    margin-bottom: 24px;
}
.app-header h1 { color: #E50914; font-size: 2.4rem; margin: 0; }
.app-header p  { color: #AAAAAA; font-size: 1rem; margin: 4px 0 0; }
.gr-button-primary {
    background: #E50914 !important;
    border: none !important;
    color: #FFFFFF !important;
    font-weight: 700 !important;
    border-radius: 6px !important;
}
.gr-button-primary:hover { background: #b20710 !important; }
textarea {
    background: #1e1e1e !important;
    border: 1px solid #333 !important;
    color: #FFFFFF !important;
    border-radius: 6px !important;
}
textarea:focus { border-color: #E50914 !important; }
#status-box textarea {
    font-family: 'Courier New', monospace !important;
    font-size: 0.82rem !important;
    color: #88cc88 !important;
    background: #0d0d0d !important;
}
"""

model_label = f'Using: {ACTIVE_MODEL}'

EXAMPLE_PROMPTS = [
    ['A lone detective standing under a neon sign in a rain-soaked Tokyo alley at midnight',
     '🌆 Neo-Noir Thriller', '🖼️  Poster (1024x1792 - Vertical)', '⚡ Standard (Faster)'],
    ['An astronaut discovering an ancient alien temple on a red desert planet with twin suns',
     '🚀 Sci-Fi Epic', '🎞️  Banner (1792x1024 - Horizontal)', '⚡ Standard (Faster)'],
    ['A powerful queen standing at the edge of a mystical glowing forest at dawn',
     '👑 Fantasy & Adventure', '🖼️  Poster (1024x1792 - Vertical)', '⚡ Standard (Faster)'],
    ['Two strangers sharing an umbrella on a rainy Parisian bridge at golden hour',
     '💘 Romantic Drama', '⬛  Square  (1024x1024)', '⚡ Standard (Faster)'],
    ['A sleek promotional banner for a documentary about the future of AI',
     '🎨 Minimalist Modern', '🎞️  Banner (1792x1024 - Horizontal)', '⚡ Standard (Faster)'],
]

with gr.Blocks(css=CUSTOM_CSS, title='Netflix AI Design Studio') as demo:

    gr.HTML(f"""
    <div class='app-header'>
        <h1>🎬 Netflix AI Design Studio</h1>
        <p>Generate bespoke banners &amp; posters for Netflix campaigns — powered by {ACTIVE_MODEL}</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### ✍️ Describe Your Design')
            prompt_input = gr.Textbox(
                label='Image Prompt',
                placeholder='e.g. A lone hero silhouetted against a burning city skyline at dusk...',
                lines=5, max_lines=10,
            )
            with gr.Accordion('🎛️ Advanced Options', open=True):
                style_dropdown = gr.Dropdown(
                    choices=list(STYLE_PRESETS.keys()),
                    value='🎬 Netflix Original (Cinematic)',
                    label='Style Preset',
                )
                size_dropdown = gr.Dropdown(
                    choices=list(SIZE_MAP_DALLE3.keys()),
                    value='🖼️  Poster (1024x1792 - Vertical)',
                    label='Output Size',
                )
                quality_dropdown = gr.Dropdown(
                    choices=list(QUALITY_MAP.keys()),
                    value='⚡ Standard (Faster)',
                    label=f'Image Quality ({"DALL-E 3 only" if ACTIVE_MODEL == "dall-e-2" else "active"})',
                )
            generate_btn = gr.Button('🎨 Generate Design', variant='primary', size='lg')
            status_box = gr.Textbox(
                label='⚙️ Generation Log',
                lines=8, max_lines=12,
                interactive=False,
                elem_id='status-box',
                placeholder='Status will appear here after generation...',
            )
        with gr.Column(scale=1):
            gr.Markdown('### 🖼️ Generated Design')
            image_output = gr.Image(
                label='Output',
                type='pil',
                show_download_button=True,
                height=600,
            )

    gr.Markdown('---\n### 💡 Example Prompts — Click to Load')
    gr.Examples(
        examples=EXAMPLE_PROMPTS,
        inputs=[prompt_input, style_dropdown, size_dropdown, quality_dropdown],
        label='Netflix Campaign Presets',
    )

    gr.HTML("""
    <div style='text-align:center;padding:20px;color:#555;font-size:0.8rem;
                border-top:1px solid #333;margin-top:24px;'>
        Netflix AI Design Studio | Powered by OpenAI DALL-E &amp; Gradio | Course-End Project
    </div>
    """)

    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_input, style_dropdown, size_dropdown, quality_dropdown],
        outputs=[image_output, status_box],
    )

print('🚀 Launching Netflix AI Design Studio...')
demo.launch(share=True, debug=False, show_error=True)

---
## 🧪 Step 7 — Programmatic Test (No UI Required)

Run this cell to test the backend directly and verify your API key is working.

In [ ]:
# STEP 7 — Backend test
TEST_PROMPT  = 'A dramatic Netflix poster of a lone astronaut floating above a dying Earth'
TEST_STYLE   = '🚀 Sci-Fi Epic'
TEST_SIZE    = '🖼️  Poster (1024x1792 - Vertical)'
TEST_QUALITY = '⚡ Standard (Faster)'

print('🧪 Running backend test...')
print(f'   Model   : {ACTIVE_MODEL}')
print(f'   Prompt  : {TEST_PROMPT}')
print()

test_image, test_status = generate_image(TEST_PROMPT, TEST_STYLE, TEST_SIZE, TEST_QUALITY)
print(test_status)

if test_image:
    os.makedirs('samples', exist_ok=True)
    save_path = f'samples/test_output_{int(time.time())}.jpg'
    test_image.save(save_path, 'JPEG', quality=95)
    print(f'\n💾 Image saved to: {save_path}')
    display(test_image)
else:
    print('\n❌ Test failed — no image returned.')

---
## 📚 Project Summary

| Component | Detail |
|-----------|--------|
| **AI Model** | DALL-E 3 (auto-falls back to DALL-E 2) |
| **UI Framework** | Gradio 4.x (`gr.Blocks`) |
| **Image Processing** | Pillow (PIL) |
| **HTTP Client** | `requests` |
| **API Key Security** | Colab Secrets → Env Var → `getpass` |
| **Sharing** | Gradio `share=True` (72-hour public link) |
| **Style Presets** | 7 Netflix-themed cinematic styles |

### 💰 Cost Reference
| Model | Size | Cost/image |
|-------|------|------------|
| DALL-E 3 Standard | 1024×1024 | ~$0.040 |
| DALL-E 3 Standard | 1792×1024 | ~$0.080 |
| DALL-E 2 | 1024×1024 | ~$0.020 |

> Monitor usage at https://platform.openai.com/usage

In [ ]:
# ── Check available models on your account ──
from openai import OpenAI
import os

try:
    from google.colab import userdata
    key = userdata.get('OPENAI_API_KEY')
except:
    key = os.environ.get('OPENAI_API_KEY', '')

client = OpenAI(api_key=key)

print("🔍 Fetching all models available on your account...\n")
models = client.models.list()
image_related = []
all_models = []

for m in models.data:
    all_models.append(m.id)
    if any(x in m.id.lower() for x in ['dall', 'image', 'gpt-image']):
        image_related.append(m.id)

print("🖼️  Image-capable models found:")
if image_related:
    for m in sorted(image_related):
        print(f"   ✅ {m}")
else:
    print("   ❌ None found with 'dall' or 'image' in name")

print()
print("📋 All available models:")
for m in sorted(all_models):
    print(f"   {m}")